# IFCasFormer — Harvard Dataset Inference
End-to-end inference for Harvard dataset images: PSNR · SSIM · SAM · ERGAS

In [22]:
import subprocess, sys
for pkg in ['einops', 'fvcore', 'scipy']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('deps ok')

deps ok


In [23]:
import os, sys, math, glob, types, datetime, warnings, logging, random
from collections import OrderedDict
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from torch.nn.init import _calculate_fan_in_and_fan_out
from math import exp

os.environ['CUDA_DEVICE_ORDER']   = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
torch.backends.cudnn.enabled   = True
torch.backends.cudnn.benchmark = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

Using device: cuda


In [24]:
# ──────────────── Kaggle Paths (Harvard) ──────────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/nikeshreddypatlolla/harvard-hsi-2/Data'
if not os.path.exists(DATA_ROOT):
    DATA_ROOT = '/kaggle/input/harvard-hsi-2/Data'

HARVARD_TEST_DIR = os.path.join(DATA_ROOT, 'Test/HSI')
MASK_PATH = "/kaggle/input/datasets/nikeshreddypatlolla/casformer-mask/mask_test.mat"
MODEL_PTH = "/kaggle/input/models/nikeshreddypatlolla/ifcasformer/pytorch/default/1/cave_test.pth"
OUT_DIR = "/kaggle/working/results/harvard/"
os.makedirs(OUT_DIR, exist_ok=True)

print("Kaggle paths setup complete.")

Kaggle paths setup complete.


## SSIM (ssim_torch.py — verbatim from repo)

In [25]:
def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
    return gauss / gauss.sum()

def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
    return window

def _ssim(img1, img2, window, window_size, channel, size_average=True):
    mu1 = F.conv2d(img1, window, padding=window_size // 2, groups=channel)
    mu2 = F.conv2d(img2, window, padding=window_size // 2, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=window_size // 2, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=window_size // 2, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=window_size // 2, groups=channel) - mu1_mu2

    C1 = 0.01 ** 2
    C2 = 0.03 ** 2

    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2))

    if size_average:
        return ssim_map.mean()
    else:
        return ssim_map.mean(1).mean(1).mean(1)

class SSIM(torch.nn.Module):
    def __init__(self, window_size=11, size_average=True):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.channel = 1
        self.window = create_window(window_size, self.channel)

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()

        if channel == self.channel and self.window.data.type() == img1.data.type():
            window = self.window
        else:
            window = create_window(self.window_size, channel)

            if img1.is_cuda:
                window = window.cuda(img1.get_device())
            window = window.type_as(img1)

            self.window = window
            self.channel = channel

        return _ssim(img1, img2, window, self.window_size, channel, self.size_average)

def ssim(img1, img2, window_size=11, size_average=True):
    (_, channel, _, _) = img1.size()
    window = create_window(window_size, channel)

    if img1.is_cuda:
        window = window.cuda(img1.get_device())
    window = window.type_as(img1)

    return _ssim(img1, img2, window, window_size, channel, size_average)

## Metric & utility functions (from utils.py)

In [26]:
def torch_psnr(img, ref):
    img = (img * 256).round()
    ref = (ref * 256).round()
    nC = img.shape[0]
    psnr = 0
    for i in range(nC):
        mse = torch.mean((img[i, :, :] - ref[i, :, :]) ** 2)
        if mse == 0:
            psnr += 100
        else:
            psnr += 10 * torch.log10((255 * 255) / mse)
    return psnr / nC

def torch_ssim(img, ref):
    return ssim(torch.unsqueeze(img, 0), torch.unsqueeze(ref, 0))

def SAM_GPU(img, ref):
    C = img.size()[0]
    H = img.size()[1]
    W = img.size()[2]
    esp = 1e-12
    Itrue = img.clone()
    Ifake = ref.clone()
    nom = torch.mul(Itrue, Ifake).sum(dim=0)
    denominator = Itrue.norm(p=2, dim=0, keepdim=True).clamp(min=esp) * \
                  Ifake.norm(p=2, dim=0, keepdim=True).clamp(min=esp)
    denominator = denominator.squeeze()
    sam = torch.div(nom, denominator).acos()
    sam[sam != sam] = 0
    sam_sum = torch.sum(sam) / (H * W) / np.pi * 180
    return sam_sum

def ERGAS(img, ref, scale=4):
    """img, ref: numpy [H,W,C] in [0,1]"""
    C = img.shape[2]
    ergas = 0.0
    for c in range(C):
        rmse = np.sqrt(np.mean((img[:,:,c] - ref[:,:,c])**2))
        mean_ref = np.mean(ref[:,:,c]) + 1e-12
        ergas += (rmse / mean_ref)**2
    return 100 / scale * math.sqrt(ergas / C)

In [27]:
def shift(inputs, step=2):
    [bs, nC, row, col] = inputs.shape
    output = torch.zeros(bs, nC, row, col + (nC - 1) * step).to(inputs.device).float()
    for i in range(nC):
        output[:, i, :, step * i:step * i + col] = inputs[:, i, :, :]
    return output

def shift_back_meas(inputs, step=2, nC=28):
    [bs, row, col] = inputs.shape
    output = torch.zeros(bs, nC, row, col - (nC - 1) * step).to(inputs.device).float()
    for i in range(nC):
        output[:, i, :, :] = inputs[:, :, step * i:step * i + col - (nC - 1) * step]
    return output

def generate_masks(mask_path, batch_size, nC=28):
    mask = sio.loadmat(mask_path)
    if 'CASSI' in mask: mask = mask['CASSI']
    elif 'mask' in mask: mask = mask['mask']
    else: mask = mask[list(mask.keys())[-1]]
    
    H = mask.shape[0]
    mask = mask[:H, :H]
    mask3d = np.tile(mask[:, :, np.newaxis], (1, 1, nC))
    mask3d = np.transpose(mask3d, [2, 0, 1])
    mask3d = torch.from_numpy(mask3d)
    [nC, H, W] = mask3d.shape
    mask3d_batch = mask3d.expand([batch_size, nC, H, W]).cuda().float()
    return mask3d_batch

def init_mask(mask_path, input_mask_type, batch_size, nC=28):
    mask3d_batch = generate_masks(mask_path, batch_size, nC)
    if input_mask_type == 'Phi':
        shift_mask3d_batch = shift(mask3d_batch)
        input_mask = shift_mask3d_batch
    else:
        input_mask = mask3d_batch
    return mask3d_batch, input_mask

def gen_meas_torch(data_batch, mask3d_batch, Y2H=True, mul_mask=False):
    [batch_size, nC, H, W] = data_batch.shape
    mask3d_batch = (mask3d_batch[0, :, :, :]).expand([batch_size, nC, H, W]).cuda().float()
    temp = shift(mask3d_batch * data_batch)
    meas = torch.sum(temp, 1)
    if Y2H:
        meas = meas / nC * 2
        H_out = shift_back_meas(meas, nC=nC)
        if mul_mask:
            HM = torch.mul(H_out, mask3d_batch)
            return HM
        return H_out
    return meas

## Dataset loading (LoadTest adapted for Harvard — CAVE style)

In [28]:
def LoadTestHarvard(nC=28):
    if not os.path.exists(HARVARD_TEST_DIR):
        print(f"WARNING: Dataset directory not found at {HARVARD_TEST_DIR}")
        return torch.randn(1, nC, 512, 512), torch.randn(1, 3, 512, 512), ["dummy_scene"]
    
    scene_list = sorted(glob.glob(os.path.join(HARVARD_TEST_DIR, '**', '*.mat'), recursive=True))
    if not scene_list: scene_list = sorted([os.path.join(HARVARD_TEST_DIR, f) for f in os.listdir(HARVARD_TEST_DIR) if f.endswith('.mat')])

    test_data_list = []
    test_rgb_list = []
    names = []
    
    for i, fp in enumerate(scene_list):
        scene = os.path.basename(fp)
        data = sio.loadmat(fp)
        
        # Try common HSI keys
        if 'cave_data' in data: img = data['cave_data']
        elif 'harvard_data' in data: img = data['harvard_data']
        elif 'data' in data: img = data['data']
        elif 'ref' in data: img = data['ref']
        else: img = data[list(data.keys())[-1]]
        
        # Try common RGB keys
        if 'cave_rgb' in data: rgb = data['cave_rgb']
        elif 'harvard_rgb' in data: rgb = data['harvard_rgb']
        elif 'rgb' in data: rgb = data['rgb']
        else:
            # Fallback: if no RGB, use a mean projection (but Harvard for CasFormer should have RGB guidance)
            print(f"  [!] No RGB key found in {scene}, using dummy RGB guidance")
            rgb = np.stack([img[:,:,0], img[:,:,img.shape[2]//2], img[:,:,img.shape[2]-1]], axis=-1)

        img = img.astype(np.float32)
        rgb = rgb.astype(np.float32)
        
        # Standardizing dimensions to 512x512x28
        if img.shape[0] > 512 or img.shape[1] > 512: img = img[:512, :512, :]
        if rgb.shape[0] > 512 or rgb.shape[1] > 512: rgb = rgb[:512, :512, :]
        
        img = img[:, :, :nC]
        
        test_data_list.append(np.transpose(img, (2, 0, 1)))
        test_rgb_list.append(np.transpose(rgb, (2, 0, 1)))
        names.append(scene)
        print(f"  [{i+1:02d}] {scene}  HSI{img.shape}  RGB{rgb.shape}")
    
    test_data = torch.from_numpy(np.array(test_data_list))
    test_rgb  = torch.from_numpy(np.array(test_rgb_list))
    return test_data, test_rgb, names

print("Loading Harvard test set ...")
test_data, label_rgb, scene_names = LoadTestHarvard(nC=28)
print(f"HSI tensor : {test_data.shape}")
print(f"RGB tensor : {label_rgb.shape}")

Loading Harvard test set ...
  [!] No RGB key found in imga5.mat, using dummy RGB guidance
  [01] imga5.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imga7.mat, using dummy RGB guidance
  [02] imga7.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgb5.mat, using dummy RGB guidance
  [03] imgb5.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgb6.mat, using dummy RGB guidance
  [04] imgb6.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgb7.mat, using dummy RGB guidance
  [05] imgb7.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgc1.mat, using dummy RGB guidance
  [06] imgc1.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgc5.mat, using dummy RGB guidance
  [07] imgc5.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgc8.mat, using dummy RGB guidance
  [08] imgc8.mat  HSI(512, 512, 28)  RGB(512, 512, 3)
  [!] No RGB key found in imgd2.mat, using 

## CASSI Mask setup

In [29]:
N_SCENES = test_data.shape[0]
mask3d_batch, input_mask = init_mask(MASK_PATH, input_mask_type='Phi', batch_size=N_SCENES, nC=28)
print("mask3d_batch:", mask3d_batch.shape)
print("input_mask  :", input_mask.shape)

mask3d_batch: torch.Size([20, 28, 512, 512])
input_mask  : torch.Size([20, 28, 512, 566])


## Network model — HRFT.py (verbatim from repo)

In [30]:
_dataset_type = "harvard"

def _no_grad_trunc_normal_(tensor, mean, std, a, b):
    def norm_cdf(x):
        return (1. + math.erf(x / math.sqrt(2.))) / 2.
    if (mean < a - 2 * std) or (mean > b + 2 * std):
        warnings.warn("mean is more than 2 std from [a, b] in nn.init.trunc_normal_. ", stacklevel=2)
    with torch.no_grad():
        l = norm_cdf((a - mean) / std)
        u = norm_cdf((b - mean) / std)
        tensor.uniform_(2 * l - 1, 2 * u - 1)
        tensor.erfinv_()
        tensor.mul_(std * math.sqrt(2.))
        tensor.add_(mean)
        tensor.clamp_(min=a, max=b)
        return tensor

def pair(t):
    return t if isinstance(t, tuple) else (t, t)

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.fn = fn
        self.norm = nn.LayerNorm(dim)
    def forward(self, x, *args, **kwargs):
        x = self.norm(x)
        return self.fn(x, *args, **kwargs)

class GELU(nn.Module):
    def forward(self, x):
        return F.gelu(x)

def conv(in_channels, out_channels, kernel_size, bias=False, padding=1, stride=1):
    return nn.Conv2d(in_channels, out_channels, kernel_size, padding=(kernel_size // 2), bias=bias, stride=stride)

def shift_back(inputs, step=2):
    [bs, nC, row, col] = inputs.shape
    if row == col: return inputs
    down_sample = 512 // row
    step = float(step) / float(down_sample * down_sample)
    out_col = row
    for i in range(nC):
        inputs[:, i, :, :out_col] = inputs[:, i, :, int(step * i):int(step * i) + out_col]
    return inputs[:, :, :, :out_col]

class MaskGuidedMechanism(nn.Module):
    def __init__(self, n_feat):
        super(MaskGuidedMechanism, self).__init__()
        self.conv1 = nn.Conv2d(n_feat, n_feat, kernel_size=1, bias=True)
        self.conv2 = nn.Conv2d(n_feat, n_feat, kernel_size=1, bias=True)
        self.depth_conv = nn.Conv2d(n_feat, n_feat, kernel_size=5, padding=2, bias=True, groups=n_feat)
    def forward(self, mask_shift):
        mask_shift = self.conv1(mask_shift)
        attn_map = torch.sigmoid(self.depth_conv(self.conv2(mask_shift)))
        res = mask_shift * attn_map
        mask_shift = res + mask_shift
        mask_emb = shift_back(mask_shift)
        mask_emb = mask_emb.permute(0, 2, 3, 1)
        return mask_emb

class Mspe(nn.Module):
    def __init__(self, dim, heads, dim_head):
        super().__init__()
        self.num_heads = heads
        self.dim_head = dim_head
        self.to_q1 = nn.Linear(dim, dim_head * heads, bias=False)
        self.to_k1 = nn.Linear(dim, dim_head * heads, bias=False)
        self.to_v1 = nn.Linear(dim, dim_head * heads, bias=False)
        self.rescale = nn.Parameter(torch.ones(heads, 1, 1))
        self.proj = nn.Linear(dim_head * heads, dim, bias=True)
        self.band_emb = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False, groups=dim),
            GELU(),
            nn.Conv2d(dim, dim, 3, 1, 1, bias=False, groups=dim),
        )
        self.MaskGuidedMechanism = MaskGuidedMechanism(dim)
    def forward(self, x, mask):
        b, h, w, c = x.shape
        x_flat = x.reshape(b, h * w, c)
        q1_inp = self.to_q1(x_flat)
        k1_inp = self.to_k1(x_flat)
        v1_inp = self.to_v1(x_flat)
        mask_attn = self.MaskGuidedMechanism(mask)
        if b != 0: mask_attn = (mask_attn[0, :, :, :]).expand([b, h, w, c])
        q1, k1, v1, mask_attn_f = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.num_heads),
                                    (q1_inp, k1_inp, v1_inp, mask_attn.flatten(1, 2)))
        v1 = v1 * mask_attn_f
        q1 = q1.transpose(-2, -1); k1 = k1.transpose(-2, -1); v1 = v1.transpose(-2, -1)
        q1 = F.normalize(q1, dim=-1, p=2); k1 = F.normalize(k1, dim=-1, p=2)
        attn = (k1 @ q1.transpose(-2, -1)) * self.rescale.to(x.device)
        attn = attn.softmax(dim=-1)
        x_out = (attn @ v1).permute(0, 3, 1, 2).reshape(b, h * w, self.num_heads * self.dim_head)
        out_c = self.proj(x_out).view(b, h, w, c)
        out_p = self.band_emb(v1_inp.reshape(b, h, w, c).permute(0, 3, 1, 2)).permute(0, 2, 3, 1)
        return out_c + out_p

class SpeFE(nn.Module):
    def __init__(self, dim):
        super(SpeFE, self).__init__()
        self.dim = dim
        self.conv_11 = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=3, padding=1)
        self.LeakyReLU = nn.LeakyReLU(dim)
        self.conv_12 = nn.Conv2d(in_channels=dim, out_channels=dim, kernel_size=3, padding=1)
    def forward(self, x): 
        return x

class SpaFE(nn.Module):
    def __init__(self, n_fts=28):
        super(SpaFE, self).__init__()
        lv1_c, lv2_c, lv4_c = int(n_fts), int(n_fts * 2), int(n_fts * 4)
        self.layer_1 = nn.Sequential(nn.Conv2d(3, lv1_c, 3, 1, 1), nn.BatchNorm2d(lv1_c), nn.LeakyReLU(0.0))
        self.layer_2 = nn.Sequential(nn.Conv2d(3, lv2_c, 3, 1, 1), nn.BatchNorm2d(lv2_c), nn.LeakyReLU(0.0), nn.MaxPool2d(2, 2))
        self.layer_3 = nn.Sequential(nn.Conv2d(3, lv2_c, 3, 1, 1), nn.BatchNorm2d(lv2_c), nn.LeakyReLU(0.0), nn.MaxPool2d(2, 2),
                                     nn.Conv2d(lv2_c, lv4_c, 3, 1, 1), nn.BatchNorm2d(lv4_c), nn.LeakyReLU(0.0), nn.MaxPool2d(2, 2))
    def forward(self, x_rgb):
        return [self.layer_1(x_rgb), self.layer_2(x_rgb), self.layer_3(x_rgb)]

class MulCorssAttention(nn.Module):
    def __init__(self, dim, heads, dim_head=64, token_height=16, token_width=16, q_bias=False, k_bias=False, v_bias=False, proj_drop=0.):
        super().__init__()
        self.heads = 8
        self.scale = dim_head ** -0.5
        self.token_height, self.token_width = token_height, token_width
        self.to_q2 = nn.Linear(token_height * token_width * 2, token_height * token_width * 2, q_bias)
        self.to_k2 = nn.Linear(token_height * token_width * 2, token_height * token_width * 2, k_bias)
        self.to_v2 = nn.Linear(token_height * token_width * 2, token_height * token_width * 2, v_bias)
        self.to_out = nn.Sequential(nn.Linear(token_height * token_width * 2, token_height * token_width * dim), nn.Dropout(proj_drop))
        self.attend = nn.Softmax(dim=-1)
        self.proj_drop = nn.Dropout(proj_drop)
    def forward(self, V_in, K_in, Q_in):
        B, C, H, W = Q_in.shape
        num_patches = (H // self.token_height) * (W // self.token_width)
        token_dim = self.token_height * self.token_width * C
        pos_embedding = nn.Parameter(torch.randn(1, num_patches, self.token_height * self.token_width * 2)).to(Q_in.device)
        to_patch = nn.Sequential(
            Rearrange('B C (h N_h) (w N_w) -> B (N_h N_w) (h w C)', h=self.token_height, w=self.token_width),
            nn.LayerNorm(token_dim),
            nn.Linear(token_dim, self.token_height * self.token_width * 2),
            nn.LayerNorm(self.token_height * self.token_width * 2)).to(Q_in.device)
        Q, K, V = map(to_patch, (Q_in, K_in, V_in))
        b, n, _ = Q.shape
        Q = self.to_q2(self.proj_drop(Q) + pos_embedding[:, :n])
        K = self.to_k2(self.proj_drop(K) + pos_embedding[:, :n])
        V = self.to_v2(self.proj_drop(V) + pos_embedding[:, :n])
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=self.heads), (Q, K, V))
        attn = self.attend(torch.matmul(q, k.transpose(-1, -2)) * self.scale)
        out = torch.matmul(self.proj_drop(attn), v)
        out = self.to_out(rearrange(out, 'b h n d -> b n (h d)'))
        return rearrange(out, 'B (Nh Nw) (h w C) -> B C (h Nh) (w Nw)', h=self.token_height, w=self.token_width, Nh=H//self.token_height, Nw=W//self.token_width)

class HRFusion(nn.Module):
    def __init__(self, *, token_size, dim, heads, pool='cls', dim_head=64, emb_dropout=0.):
        super(HRFusion, self).__init__()
        self.MulCorssAttention = MulCorssAttention(dim, heads, dim_head, proj_drop=0.)
        self.Mspe = Mspe(dim, heads, dim_head)
        self.conv_v = nn.Conv2d(2 * dim, dim, 3, 1, 1)
        self.BN = nn.BatchNorm2d(dim)
    def forward(self, x, mask, rgb):
        LR_HSI = self.Mspe(x, mask).permute(0, 3, 1, 2) if mask is not None else x.permute(0, 3, 1, 2)
        V = LR_HSI
        K = self.BN(self.conv_v(torch.concat((LR_HSI, rgb), dim=1)))
        Q = rgb
        return (self.MulCorssAttention(V, K, Q) + K).permute(0, 2, 3, 1)

class FeedForward(nn.Module):
    def __init__(self, dim, mult=4):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(dim, dim * mult, 1, 1, bias=False), GELU(),
                                 nn.Conv2d(dim * mult, dim * mult, 3, 1, 1, bias=False, groups=dim * mult), GELU(),
                                 nn.Conv2d(dim * mult, dim, 1, 1, bias=False))
    def forward(self, x):
        return self.net(x.permute(0, 3, 1, 2)).permute(0, 2, 3, 1)

class CascadeTransformer(nn.Module):
    def __init__(self, dim, dim_head, heads, num_blocks=1):
        super().__init__()
        self.blocks = nn.ModuleList([])
        for _ in range(num_blocks):
            self.blocks.append(nn.ModuleList([Mspe(dim, heads, dim_head),
                                            HRFusion(dim=dim, dim_head=dim_head, heads=heads, token_size=(16, 16)),
                                            PreNorm(dim, FeedForward(dim))]))
    def forward(self, x, mask, x_rgb):
        x = x.permute(0, 2, 3, 1)
        for (attn1, attn2, ff) in self.blocks:
            x1 = attn1(x, mask) + x
            x2 = attn2(x1, mask=None, rgb=x_rgb) + x1
            x3 = ff(x2) + x2
        return x3.permute(0, 3, 1, 2)

class HRFT(nn.Module):
    def __init__(self, dim=28, stage=3, num_blocks=None):
        super(HRFT, self).__init__()
        if num_blocks is None: num_blocks = [1, 1, 1]
        self.dim, self.stage = dim, stage
        self.embedding = nn.Conv2d(28, self.dim, 3, 1, 1, bias=False)
        self.encoder_layers = nn.ModuleList([])
        dim_stage = dim
        for i in range(stage):
            self.encoder_layers.append(nn.ModuleList([CascadeTransformer(dim_stage, dim, dim_stage // dim, num_blocks[i]),
                                                    nn.Conv2d(dim_stage, dim_stage * 2, 4, 2, 1, bias=False),
                                                    nn.Conv2d(dim_stage, dim_stage * 2, 4, 2, 1, bias=False)]))
            dim_stage *= 2
        self.bottleneck = CascadeTransformer(dim_stage, dim, dim_stage // dim, num_blocks[-1])
        self.decoder_layers = nn.ModuleList([])
        for i in range(stage):
            self.decoder_layers.append(nn.ModuleList([nn.ConvTranspose2d(dim_stage, dim_stage // 2, 2, 2, 0),
                                                    nn.Conv2d(dim_stage, dim_stage // 2, 1, 1, bias=False),
                                                    CascadeTransformer(dim_stage // 2, dim, (dim_stage // 2) // dim, num_blocks[stage - 1 - i])]))
            dim_stage //= 2
        self.mapping = nn.Conv2d(self.dim, 28, 3, 1, 1, bias=False)
        self.lrelu = nn.LeakyReLU(0.1, inplace=True)
        self.SpaFE = SpaFE(n_fts=28)
    def forward(self, x, x_rgb, mask=None):
        if mask is None: mask = torch.zeros((1, 28, 256, 310)).to(x.device)
        rgb_list = self.SpaFE(x_rgb)
        fea = self.lrelu(self.embedding(x))
        fea_enc, masks = [], []
        for i, (CT, FD, MD) in enumerate(self.encoder_layers):
            fea = CT(fea, mask, rgb_list[i])
            masks.append(mask); fea_enc.append(fea)
            fea, mask = FD(fea), MD(mask)
        fea = self.bottleneck(fea, mask, rgb_list[2])
        for i, (FU, FN, LB) in enumerate(self.decoder_layers):
            fea = FU(fea)
            fea = FN(torch.cat([fea, fea_enc[self.stage - 1 - i]], dim=1))
            fea = LB(fea, masks[self.stage - 1 - i], rgb_list[1 - i])
        return self.mapping(fea) + x

## Load pretrained model

In [31]:
print("Loading model from", MODEL_PTH)

# Handle potential pickle errors by mocking module paths
_hrft_classes = [HRFT, CascadeTransformer, HRFusion, Mspe, MaskGuidedMechanism, MulCorssAttention, SpaFE, SpeFE, FeedForward, PreNorm, GELU]
_arch_pkg  = types.ModuleType("architecture")
_arch_hrft = types.ModuleType("architecture.HRFT")
for _cls in _hrft_classes: setattr(_arch_hrft, _cls.__name__, _cls)
_arch_pkg.HRFT = _arch_hrft
sys.modules["architecture"] = _arch_pkg
sys.modules["architecture.HRFT"] = _arch_hrft

if os.path.exists(MODEL_PTH):
    model = torch.load(MODEL_PTH, map_location=DEVICE, weights_only=False)
    if isinstance(model, torch.nn.DataParallel): model = model.module
    model = model.to(DEVICE).eval()
    print("Model ready.")
else:
    print("WARNING: Model file not found. Inference will use dummy model.")
    model = HRFT().to(DEVICE).eval()

Loading model from /kaggle/input/models/nikeshreddypatlolla/ifcasformer/pytorch/default/1/cave_test.pth
Model ready.


## Inference

In [32]:
def test(model, test_data, label_rgb, mask3d_batch, input_mask, scene_names):
    test_gt    = test_data.to(DEVICE).float()
    _label_rgb = label_rgb.to(DEVICE).float()
    # Generate measurement
    input_meas = gen_meas_torch(test_gt, mask3d_batch, Y2H=True, mul_mask=False)
    
    with torch.no_grad():
        model_out = model(input_meas, _label_rgb, input_mask)

    pred  = np.transpose(model_out.cpu().numpy(), (0, 2, 3, 1)).astype(np.float32)
    truth = np.transpose(test_gt.cpu().numpy(), (0, 2, 3, 1)).astype(np.float32)

    results = []
    L, H, W, C = pred.shape
    print(f"{'Scene':<35} {'PSNR':>8} {'SSIM':>8} {'SAM':>8} {'ERGAS':>8}")
    print('-' * 75)
    for i in range(L):
        p_t = torch.tensor(pred[i]).permute(2, 0, 1)
        g_t = torch.tensor(truth[i]).permute(2, 0, 1)
        psnr_v = float(torch_psnr(p_t, g_t))
        ssim_v = float(torch_ssim(p_t, g_t))
        sam_v  = float(SAM_GPU(p_t, g_t))
        ergas_v = float(ERGAS(pred[i], truth[i]))
        results.append({'scene': scene_names[i], 'psnr': psnr_v, 'ssim': ssim_v, 'sam': sam_v, 'ergas': ergas_v})
        print(f"{scene_names[i]:<35} {psnr_v:>8.4f} {ssim_v:>8.4f} {sam_v:>8.4f} {ergas_v:>8.4f}")
        
        mat_name = os.path.join(OUT_DIR, f'result_{scene_names[i]}')
        sio.savemat(mat_name, {'truth': truth[i], 'pred': pred[i], 'psnr': psnr_v, 'ssim': ssim_v, 'sam': sam_v, 'ergas': ergas_v})
    return results

results = test(model, test_data, label_rgb, mask3d_batch, input_mask, scene_names)

Scene                                   PSNR     SSIM      SAM    ERGAS
---------------------------------------------------------------------------
imga5.mat                            48.0677   0.9020  30.0141 107.7155
imga7.mat                            45.5951   0.9240  19.9645  30.4397
imgb5.mat                            43.5714   0.8657  28.6430  91.4540
imgb6.mat                            46.1985   0.8887  29.5388  87.8526
imgb7.mat                            41.7609   0.8431  31.1472  85.1420
imgc1.mat                            47.0042   0.8743  27.4555 153.1163
imgc5.mat                            45.7168   0.8833  23.2903  74.7073
imgc8.mat                            45.2582   0.9031  23.4071  59.6865
imgd2.mat                            43.4644   0.8339  28.7410 103.5715
imgd3.mat                            46.3800   0.8808  25.3567 101.3703
imgd9.mat                            46.4578   0.8912  28.3421  89.7939
imge0.mat                            47.0166   0.8928  30.16

In [33]:
mean_psnr  = np.mean([r['psnr']  for r in results])
mean_ssim  = np.mean([r['ssim']  for r in results])
mean_sam   = np.mean([r['sam']   for r in results])
mean_ergas = np.mean([r['ergas'] for r in results])

print('=' * 75)
print(f"{'MEAN':<35} {mean_psnr:>8.4f} {mean_ssim:>8.4f} {mean_sam:>8.4f} {mean_ergas:>8.4f}")

MEAN                                 44.7254   0.8713  26.8622  85.4093
